# Stage 3 — Fine-tune DistilBERT (Colab, GPU)

Implements section 6 of `unfair-tos-architecture-plan.md`: fine-tune `distilbert-base-uncased` for multi-label classification on the same UNFAIR-ToS splits used for the baseline.

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better). This notebook re-downloads the dataset directly (Colab has full internet access), so you don't need to re-upload the parquet files from Stage 1–2 — it's self-contained. Run All.

## 0. Setup

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas

# Colab sometimes ships a torch/torchvision version mismatch that breaks datasets'
# torch-tensor formatting (ImportError: cannot import name 'VideoReader' from
# 'torchvision.io') even though this notebook never uses any vision features.
# Removing torchvision avoids that code path entirely.
!pip uninstall -y -q torchvision torchaudio

In [ ]:
import os, time, shutil, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset, Dataset
import datasets

# Belt-and-suspenders fix for a Colab torch/torchvision version mismatch that otherwise
# crashes datasets' torch-tensor formatting with:
#   ImportError: cannot import name 'VideoReader' from 'torchvision.io'
# This notebook never touches images/video, so we force the flag off directly rather
# than relying on torchvision being fully uninstalled (which needs a runtime restart
# to take effect, and is easy to forget).
datasets.config.TORCHVISION_AVAILABLE = False

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from sklearn.metrics import f1_score, classification_report

CATEGORIES = [
    "Limitation of liability", "Unilateral termination", "Unilateral change",
    "Content removal", "Contract by using", "Choice of law",
    "Jurisdiction", "Arbitration",
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU detected — go to Runtime > Change runtime type > T4 GPU, "
          "or training will be very slow.")

os.makedirs("models/distilbert", exist_ok=True)

## 1. Load data and build the multi-hot label matrix

Same loading logic as Stage 1, kept self-contained here so this notebook doesn't depend on uploading intermediate files.

In [ ]:
raw = load_dataset("coastalcph/lex_glue", "unfair_tos")

def to_frame(split):
    df = split.to_pandas()
    for i, cat in enumerate(CATEGORIES):
        df[cat] = df["labels"].apply(lambda ls: int(i in ls))
    return df

train_df = to_frame(raw["train"])
val_df   = to_frame(raw["validation"])
test_df  = to_frame(raw["test"])

print(f"train: {len(train_df):,} | val: {len(val_df):,} | test: {len(test_df):,}")

## 2. Pick `max_length` from the tokenizer's actual token-count distribution

Stage 1's EDA looked at raw word counts, which run shorter than subword-tokenized length. Since we have the tokenizer here anyway, it's more accurate to measure real token counts directly rather than guess a word-to-token multiplier.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_lengths = [len(tokenizer.encode(t, add_special_tokens=True)) for t in train_df["text"]]
sample_lengths = np.array(sample_lengths)

for p in [50, 75, 90, 95, 99, 100]:
    print(f"  p{p}: {np.percentile(sample_lengths, p):.0f} tokens")

# Round up to a clean power-of-two-ish value that comfortably covers the 99th percentile
p99 = np.percentile(sample_lengths, 99)
for candidate in [64, 96, 128, 192, 256]:
    if candidate >= p99:
        MAX_LENGTH = candidate
        break
else:
    MAX_LENGTH = 256

print(f"\nSelected max_length = {MAX_LENGTH} (covers {(sample_lengths <= MAX_LENGTH).mean()*100:.1f}% of train sentences without truncation)")

## 3. Tokenize and build HF Datasets

In [ ]:
def to_hf_dataset(df):
    labels = df[CATEGORIES].values.astype(np.float32).tolist()
    ds = Dataset.from_dict({"text": df["text"].tolist(), "labels": labels})
    return ds

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds = to_hf_dataset(train_df).map(tokenize, batched=True)
val_ds   = to_hf_dataset(val_df).map(tokenize, batched=True)
test_ds  = to_hf_dataset(test_df).map(tokenize, batched=True)

keep_cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format("torch", columns=keep_cols)
val_ds.set_format("torch", columns=keep_cols)
test_ds.set_format("torch", columns=keep_cols)

print(train_ds)

## 4. Per-category `pos_weight` for the loss

`class_weight="balanced"` (used in the sklearn baseline) has no direct equivalent for `BCEWithLogitsLoss`. `pos_weight` is the analogous idea: it upweights the loss on positive examples for categories where negatives dominate, computed here as `negatives / positives` per category from the training set.

In [ ]:
USE_POS_WEIGHT = True  # flip to False to compare against a plain BCE loss

pos_counts = train_df[CATEGORIES].sum().values
neg_counts = len(train_df) - pos_counts
pos_weight_values = neg_counts / np.maximum(pos_counts, 1)  # avoid div-by-zero
pos_weight = torch.tensor(pos_weight_values, dtype=torch.float32).to(device)

for cat, w in zip(CATEGORIES, pos_weight_values):
    print(f"  {cat}: pos_weight={w:.1f}")

## 5. Model, WeightedTrainer, and metrics

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(CATEGORIES),
    problem_type="multi_label_classification",
).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight if USE_POS_WEIGHT else None)
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs >= 0.5).astype(int)  # fixed 0.5 here; Stage 4 tunes per-category thresholds properly
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
    }

## 6. Train

In [ ]:
args = TrainingArguments(
    output_dir="distilbert-unfair-tos-checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,          # keep disk usage down — only best + latest checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

t0 = time.time()
trainer.train()
print(f"\nTraining took {(time.time()-t0)/60:.1f} min")

`load_best_model_at_end=True` restores the checkpoint with the highest validation `f1_macro` across all epochs once training finishes — the model in memory after this cell is already the best one, not necessarily the last epoch's.

## 7. Evaluate on validation — same report format as Stage 2, for direct comparison

In [ ]:
val_output = trainer.predict(val_ds)
val_probs = 1 / (1 + np.exp(-val_output.predictions))
val_preds = (val_probs >= 0.5).astype(int)
val_labels = val_output.label_ids

report_text = classification_report(val_labels, val_preds, target_names=CATEGORIES, zero_division=0)
report_dict = classification_report(val_labels, val_preds, target_names=CATEGORIES, zero_division=0, output_dict=True)

print(report_text)
print(f"Validation macro-F1: {f1_score(val_labels, val_preds, average='macro', zero_division=0):.4f}")
print(f"Validation micro-F1: {f1_score(val_labels, val_preds, average='micro', zero_division=0):.4f}")

**Read this against the Stage 2 baseline report (macro-F1 0.71, micro-F1 0.71 there).** These numbers still use a flat 0.5 threshold per category — Stage 4 sweeps per-category thresholds on validation probabilities for both models before the final head-to-head test-set comparison, so don't treat this as the final verdict on which model wins. It's useful here mainly to sanity-check that fine-tuning actually worked (macro-F1 comfortably above the untrained/random baseline) before moving on.

## 8. Which categories does DistilBERT struggle with?

In [ ]:
per_cat_f1 = {cat: report_dict[cat]["f1-score"] for cat in CATEGORIES}
ranked = sorted(per_cat_f1.items(), key=lambda x: x[1])

print("DistilBERT categories ranked worst -> best by validation F1:")
for cat, f1 in ranked:
    support = int(report_dict[cat]["support"])
    print(f"  {f1:.3f}  {cat}  (support={support})")

Compare this ranking against Stage 2's baseline ranking (Arbitration, Unilateral change, and Limitation of liability were the baseline's three weakest). If the same categories show up here too, that's a signal the difficulty is inherent to the data (small support, ambiguous phrasing) rather than something a stronger model architecture fixes — worth calling out explicitly in the Stage 4 write-up rather than treating every category gap as a modeling failure.

## 9. Save the model + tokenizer

In [ ]:
FINAL_DIR = "models/distilbert"
trainer.save_model(FINAL_DIR)      # saves the best checkpoint (see note above cell 6)
tokenizer.save_pretrained(FINAL_DIR)

# Clean up the larger intermediate checkpoint directory now that the final model is saved separately
shutil.rmtree("distilbert-unfair-tos-checkpoints", ignore_errors=True)

print("Saved to", FINAL_DIR)
print(os.listdir(FINAL_DIR))

---
## 10. Download everything worth keeping

The DistilBERT checkpoint is a few hundred MB — zipping and downloading may take a minute depending on your connection. If you'd rather not wait, an alternative is mounting Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) and copying `models/distilbert/` there directly instead of downloading.

In [ ]:
shutil.make_archive("stage3_distilbert", "zip", ".", "models/distilbert")
size_mb = os.path.getsize("stage3_distilbert.zip") / (1024 * 1024)
print(f"Bundled models/distilbert/ into stage3_distilbert.zip ({size_mb:.1f} MB)")

try:
    from google.colab import files
    files.download("stage3_distilbert.zip")
except ImportError:
    print("Not running in Colab — find stage3_distilbert.zip in the working directory.")